# Pore water density

Water density along the pore axis, averaged over replicas.

The pore axis passes through the $xy$ centroid of the S6 Cα atoms of the four subunits in each frame. Water oxygen atoms within 3 Å of the axis are counted in 0.5 Å slices of $z$. $z$ is measured from the centroid of the L641 Cα atoms (L521 in PKD2L1), with positive values towards the cytoplasm. All vectors use the minimum-image convention, so the raw trajectory can be used.

In [ ]:
import numpy as np
import MDAnalysis as mda
import matplotlib.pyplot as plt
from MDAnalysis.lib.distances import minimize_vectors

# settings
runs = [("r1/step7.gro", "r1/step7.xtc"),
        ("r2/step7.gro", "r2/step7.xtc"),
        ("r3/step7.gro", "r3/step7.xtc")]
water_selection = "resname OPC and name O"        # "resname TIP3 and name OH2" for CHARMM TIP3P
s6_selection = "protein and name CA and resid 663:690"
origin_residue = 641                               # 521 for PKD2L1
gate_residues = [676, 680]                         # marked on the plot, the last one points to the cytoplasm
cylinder_radius = 3.0                              # A
z_edges = np.arange(-60, 60.01, 0.5)               # A

In [ ]:
profiles = []
gate_z = {residue: [] for residue in gate_residues}
for gro_file, xtc_file in runs:
    u = mda.Universe(gro_file, xtc_file)
    water = u.select_atoms(water_selection)
    s6 = u.select_atoms(s6_selection)
    origin = u.select_atoms(f"protein and name CA and resid {origin_residue}")
    gates = {residue: u.select_atoms(f"protein and name CA and resid {residue}") for residue in gate_residues}
    counts = np.zeros(len(z_edges) - 1)
    direction = None
    for ts in u.trajectory:
        box = ts.dimensions
        anchor = s6.positions[0]
        axis_xy = anchor[:2] + minimize_vectors(s6.positions - anchor, box).mean(0)[:2]
        origin_z = anchor[2] + minimize_vectors(origin.positions - anchor, box).mean(0)[2]
        centre = np.array([axis_xy[0], axis_xy[1], origin_z])
        if direction is None:
            last_gate = gates[gate_residues[-1]]
            direction = 1.0 if minimize_vectors(last_gate.positions - centre, box).mean(0)[2] > 0 else -1.0
        vectors = minimize_vectors(water.positions - centre, box)
        radial_distance = np.hypot(vectors[:, 0], vectors[:, 1])
        z_values = direction * vectors[:, 2]
        counts += np.histogram(z_values[radial_distance < cylinder_radius], bins=z_edges)[0]
        for residue, group in gates.items():
            gate_z[residue].append(direction * minimize_vectors(group.positions - centre, box).mean(0)[2])
    slice_volume = np.pi * cylinder_radius ** 2 * (z_edges[1] - z_edges[0]) / 1000      # nm^3
    profiles.append(counts / len(u.trajectory) / slice_volume)

profiles = np.array(profiles)
mean_density = profiles.mean(0)
sd_density = profiles.std(0)
z_centres = (z_edges[1:] + z_edges[:-1]) / 2 / 10                                    # nm

In [ ]:
fig, ax = plt.subplots(figsize=(3.2, 3.5))
ax.plot(mean_density, z_centres, color="#E8837A")
ax.fill_betweenx(z_centres, mean_density - sd_density, mean_density + sd_density, color="#E8837A", alpha=0.3, linewidth=0)
for residue, values in gate_z.items():
    ax.axhline(np.mean(values) / 10, color="grey", linestyle="--", linewidth=0.8)
    ax.text(ax.get_xlim()[1], np.mean(values) / 10, str(residue), ha="right", va="bottom", fontsize=8)
ax.set_xlabel("Water density (nm$^{-3}$)")
ax.set_ylabel(f"z from residue {origin_residue} Cα (nm)")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig("pore_water_density.png", dpi=300)